In [12]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

from methods import ETL

In [13]:
#Set your variables
csv_filename = 'eia_retail_sales_mwh_monthly_state_sectorwide.csv'
path = r'C:\Users\TobyWong\Desktop\School\EIA Power Prediction\EIA_end_customer_sales_power_prediction_NC\data'
stateid = 'NC'
drop_columns = ['ALL', 'OTH', 'RES', 'TRA']
keep_columns = ['COM', 'IND']

In [14]:
df = ETL(
    csv_filename=csv_filename,
    os_path=path,
    stateid=stateid,
    drop_columns=drop_columns,
    keep_columns=keep_columns
)
df.head(1)

,period,stateid,salesUnit,COM_IND
31,2025-11-01,NC,million kilowatt hours,5773.31726


In [ ]:
df_model = df.copy()
df_model["period"] = pd.to_datetime(df_model["period"])
df_model = df_model.sort_values("period").reset_index(drop=True)
df_model["time_idx"] = np.arange(len(df_model))

# Train/test split (70/30)
split_idx = int(len(df_model) * 0.7)
train = df_model.iloc[:split_idx].copy()
test = df_model.iloc[split_idx:].copy()

X_train = train[["time_idx"]]
y_train = train["COM_IND"]

X_test = test[["time_idx"]]
y_test = test["COM_IND"]

# Fit simple linear regression
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions on test set
test["predicted_COM_IND"] = model.predict(X_test)

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, test["predicted_COM_IND"]))
r2 = r2_score(y_test, test["predicted_COM_IND"])

print("Intercept:", model.intercept_)
print("Slope:", model.coef_[0])
print("Test RMSE:", rmse)
print("Test R^2:", r2)

# Forecast next 24 months
last_date = df_model["period"].max()
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=24, freq="MS")
future_idx = np.arange(len(df_model), len(df_model) + 24)

future_df = pd.DataFrame({
    "period": future_dates,
    "time_idx": future_idx
})

future_df["predicted_COM_IND"] = model.predict(future_df[["time_idx"]])
future_df["stateid"] = "NC"
future_df["salesUnit"] = "million kilowatt hours"

print(future_df[["period", "stateid", "salesUnit", "predicted_COM_IND"]])

# Plot
plt.figure(figsize=(12, 6))
plt.plot(df_model["period"], df_model["COM_IND"], label="Actual")
plt.plot(test["period"], test["predicted_COM_IND"], label="Test Prediction")
plt.plot(future_df["period"], future_df["predicted_COM_IND"], label="24-Month Forecast")
plt.xlabel("Period")
plt.ylabel("COM_IND")
plt.title("Simple Linear Regression Forecast")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Intercept: 5985.228238818865
Slope: 1.458477234857248


NameError: name 'rmse' is not defined